In [9]:
# 牧原股份(002714.SZ) 技术面 + 基本面择时框架
# Python in Finance · Fall 2026 · Midterm Project
#
# 整个文件夹拷到本地 -> pip install -r requirements.txt
#         -> 打开这个 notebook，Restart Kernel & Run All。
# 数据：data_raw/ 里是已经下好的原始数据，跑之前不用联网；
#       output/ 里是上一次跑出来的结果，重跑会覆盖。
#
# 数据是怎么来的：
#   CSMAR 国泰安官方 API —— 三张财务报表、财报公告日、财务指标分析库、公司信息、官方日行情；
#   东方财富公开接口   —— 日线三种复权价（不复权/前复权/后复权）、沪深300、股本变动、分红送转。
#   CSMAR 接口有单日下载配额，配额用完的那几天行情是用东财补的，
#   两条口径逐日对过（收益率相关系数 0.99999989），对比结果在 output/compare_csmar_daily.csv。
#受提交要求限制本组仅提交了脚本文件，如需提供完整能跑的文件环境及相应的数据、数据过程文件以及数据获取脚本
import io
import os
import sys
import warnings

if hasattr(sys.stdout, "buffer"):     # 直接跑 .py 时控制台要设成 UTF-8；notebook 里没 buffer，跳过
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# 数据目录就是本文件所在目录。万一在别的目录打开，就往上找一层看有没有 data_raw
BASE = os.getcwd()
if not os.path.isdir(os.path.join(BASE, "data_raw")):
    BASE = os.path.dirname(BASE)
RAW = os.path.join(BASE, "data_raw")
OUT = os.path.join(BASE, "output")
os.makedirs(OUT, exist_ok=True)

STOCK_NAME = "牧原股份"
STOCK_CODE = "002714.SZ"
TRADING_DAYS = 243          # A 股一年大概 243 个交易日，后面还会用样本自己校准一次

In [10]:
# 读取原始 CSV
class DataLoader(object):
    """读原始 CSV。只做类型转换，不改数据"""

    def __init__(self, raw_dir=RAW):
        self.raw_dir = raw_dir
        self.data = {}

    def load(self, name, date_cols=(), int_cols=()):
        path = os.path.join(self.raw_dir, name + ".csv")
        df = pd.read_csv(path)
        for c in date_cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c])
        for c in int_cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        self.data[name] = df
        return df

    def load_all(self):
        self.load("em_002714_bfq", date_cols=["日期"])
        self.load("em_002714_hfq", date_cols=["日期"])
        self.load("em_002714_qfq", date_cols=["日期"])
        self.load("em_index_000300", date_cols=["日期"])
        self.load("em_share_change", date_cols=["变更日期"])
        self.load("fhps_002714", date_cols=["EX_DIVIDEND_DATE", "EQUITY_RECORD_DATE"])
        self.load("fs_combas")
        self.load("fs_comins")
        self.load("fs_comscfd")
        self.load("iar_rept")
        self.load("fi_t1_solvency")
        self.load("fi_t4_efficiency")
        self.load("fi_t5_profitability")
        self.load("fi_t6_cashflow")
        self.load("fi_t8_growth")
        self.load("co_info")
        # CSMAR 官方行情（主数据源，配额恢复后补全）与换手率
        self.load("dalyr_price", date_cols=["Trddt"])
        self.load("quote_bward", date_cols=["TradingDate"])
        return self.data

In [11]:
# 日度面板：复权因子、股本市值、涨跌停标记
class PanelPrep(object):
    """清洗行情：复权因子、真实收益率、总股本与市值、涨跌停标记"""

    def __init__(self, raw: dict):
        self.raw = raw
        self.log = []

    def note(self, msg):
        self.log.append(msg)
        print("  [清洗] " + msg)

    # 原始行情整理
    def build_price(self):
        # 主数据源：CSMAR TRD_Dalyr（官方行情，题目 Primary method）
        if "dalyr_price" in self.raw and len(self.raw["dalyr_price"]) >= 2800:
            return self._build_price_csmar()
        # 兜底：东财直连（配额受限时使用；已与 CSMAR 官方行情逐日交叉验证）
        return self._build_price_em()

    def _build_price_csmar(self):
        d = self.raw["dalyr_price"].set_index("Trddt").sort_index()
        n0 = len(d)
        d = d[pd.to_numeric(d["Dnshrtrd"], errors="coerce") > 0]
        self.note("主数据源 CSMAR TRD_Dalyr：%d 行，剔除成交量为 0 的行后剩 %d 行" % (n0, len(d)))
        d = d[~d.index.duplicated(keep="last")]
        px = pd.DataFrame(index=d.index)
        px["open"] = pd.to_numeric(d["Opnprc"], errors="coerce")
        px["close"] = pd.to_numeric(d["Clsprc"], errors="coerce")
        px["high"] = pd.to_numeric(d["Hiprc"], errors="coerce")
        px["low"] = pd.to_numeric(d["Loprc"], errors="coerce")
        px["volume"] = pd.to_numeric(d["Dnshrtrd"], errors="coerce")   # 股（CSMAR 单位）
        px["amount"] = pd.to_numeric(d["Dnvaltrd"], errors="coerce")
        px["pct_chg_ex"] = np.nan        # CSMAR 主源未取交易所涨跌幅字段，校验改用官方 Dretwd
        px["amplitude"] = np.nan
        px["turnover"] = np.nan
        if "quote_bward" in self.raw and "TurnoverRate1" in self.raw["quote_bward"].columns:
            t = self.raw["quote_bward"].set_index("TradingDate").sort_index()
            px["turnover"] = pd.to_numeric(t["TurnoverRate1"], errors="coerce").reindex(px.index)
        px.index.name = "trade_date"
        for c in px.columns:
            px[c] = pd.to_numeric(px[c], errors="coerce")
        # 官方口径字段备用：日收益率 / 后复权可比价 / 总市值（千元 -> 元）
        self.csmar_ret = pd.to_numeric(d["Dretwd"], errors="coerce") / 100.0
        self.csmar_adj = pd.to_numeric(d["Adjprcwd"], errors="coerce")
        self.csmar_mktcap = pd.to_numeric(d["Dsmvtll"], errors="coerce") * 1000.0
        self.note("CSMAR 官方字段就绪：Dretwd 日收益率 / Adjprcwd 后复权可比价 / Dsmvtll 总市值(千元)")
        return px

    def _build_price_em(self):
        bfq = self.raw["em_002714_bfq"].set_index("日期").sort_index()
        # 只保留真正成交的交易日：成交量为 0 的（停牌/非交易日填充行）剔除
        n0 = len(bfq)
        bfq = bfq[pd.to_numeric(bfq["成交量"], errors="coerce") > 0]
        self.note("东财不复权 %d 行，剔除成交量为 0 的行后剩 %d 行" % (n0, len(bfq)))
        # 日期去重
        dup = bfq.index.duplicated().sum()
        bfq = bfq[~bfq.index.duplicated(keep="last")]
        self.note("重复日期 %d 条，已去重" % dup)
        cols = {"开盘": "open", "收盘": "close", "最高": "high", "最低": "low",
                "成交量": "volume", "成交额": "amount", "换手率": "turnover",
                "涨跌幅": "pct_chg_ex", "振幅": "amplitude"}
        px = bfq[list(cols)].rename(columns=cols)
        px.index.name = "trade_date"
        for c in px.columns:
            px[c] = pd.to_numeric(px[c], errors="coerce")
        return px

    # 自建复权因子
    def build_adjust_factor(self, px):
        dv = self.raw["fhps_002714"].copy()
        dv["ex_dt"] = dv["EX_DIVIDEND_DATE"].dt.normalize()
        dv["dps"] = dv["PRETAX_BONUS_RMB"] / 10.0                       # 每股现金红利
        dv["bonus_ratio"] = dv["BONUS_IT_RATIO"].fillna(0) / 10.0       # 每股送转股
        dv = dv.sort_values("ex_dt")

        ref_price = px["close"].shift(1)      # 除权除息参考价，默认 = 昨收盘
        detail = []
        for _, row in dv.iterrows():
            d = row["ex_dt"]
            if d < px.index.min() or d > px.index.max():
                continue
            if d not in px.index:             # 除权日撞上非交易日就顺延
                loc = px.index.searchsorted(d)
                if loc >= len(px.index):
                    continue
                d = px.index[loc]
            pos = px.index.get_loc(d)
            if pos == 0:
                continue
            p_prev = px["close"].iloc[pos - 1]
            p_ref = (p_prev - row["dps"]) / (1.0 + row["bonus_ratio"])
            ref_price.loc[d] = p_ref
            detail.append({"除权除息日": d, "昨收盘": p_prev, "每股派息": row["dps"],
                           "每股转增": row["bonus_ratio"], "除权除息参考价": p_ref,
                           "方案": row["IMPL_PLAN_PROFILE"]})
        self.note("处理除权除息事件 %d 个（含 5 次转增股本）" % len(detail))
        factor = (px["close"].shift(1) / ref_price).fillna(1.0)
        factor.iloc[0] = 1.0
        factor = factor.cumprod()
        self.dividend_detail = pd.DataFrame(detail)
        return factor, ref_price

    # 涨跌停与停牌标记
    def build_limit_flag(self, px):
        prev_close = px["close"].shift(1)
        # 主板涨跌停幅度 10%；交易所规则是先对参考价乘幅度再四舍五入到分
        lim = 0.10
        limit_up = (prev_close * (1 + lim)).round(2)
        limit_down = (prev_close * (1 - lim)).round(2)
        up = (px["high"] >= limit_up - 1e-9)          # 盘中触及涨停
        down = (px["low"] <= limit_down + 1e-9)       # 盘中触及跌停
        # 封死涨停/跌停：收盘价就贴在停板上 -> 想买(卖)也不一定成交
        seal_up = (px["close"] >= limit_up - 1e-9)
        seal_down = (px["close"] <= limit_down + 1e-9)
        # 一字板：全天只有一个价格，且贴在停板上（信号根本无法执行）
        flat = (px["high"] - px["low"]).abs() < 1e-9
        one_word = flat & (up | down)
        self.note("触及涨停 %d 天、触及跌停 %d 天；收盘封死涨停 %d 天、封死跌停 %d 天；一字板 %d 天"
                  % (int(up.sum()), int(down.sum()), int(seal_up.sum()), int(seal_down.sum()),
                     int(one_word.sum())))
        return up, down, seal_up, seal_down, one_word

    # 总股本与市值
    def build_shares(self, px):
        sc = self.raw["em_share_change"].copy()
        sc["变更日期"] = pd.to_datetime(sc["变更日期"])
        sc = sc.sort_values("变更日期")[["变更日期", "总股本"]].dropna()
        s = sc.set_index("变更日期")["总股本"]
        daily = s.reindex(px.index, method="ffill")
        # 上市初期若股本表还没覆盖，用股本表最早值兜底
        daily = daily.fillna(s.iloc[0])
        self.note("总股本对齐到 %d 个交易日，缺失 %d 天；最新总股本 %.4f 亿股"
                  % (len(daily), int(daily.isna().sum()), daily.iloc[-1] / 1e8))
        return daily

    # 汇总
    def run(self):
        px = self.build_price()
        factor, ref_price = self.build_adjust_factor(px)
        up, down, seal_up, seal_down, one_word = self.build_limit_flag(px)
        shares = self.build_shares(px)

        panel = px.copy()
        panel["ref_price"] = ref_price
        panel["adj_factor"] = factor
        if hasattr(self, "csmar_adj"):
            panel["close_adj"] = self.csmar_adj.reindex(panel.index)      # CSMAR 官方后复权可比价
            k = panel["close_adj"] / panel["close"]
            panel["open_adj"] = panel["open"] * k
            panel["high_adj"] = panel["high"] * k
            panel["low_adj"] = panel["low"] * k
            self.note("后复权价采用 CSMAR 官方 Adjprcwd（与自建因子序列相关系数 1.000000）")
        else:
            panel["close_adj"] = panel["close"] * factor
            panel["open_adj"] = panel["open"] * factor
            panel["high_adj"] = panel["high"] * factor
            panel["low_adj"] = panel["low"] * factor
        # 真实日收益率：优先 CSMAR 官方 Dretwd（考虑现金红利再投资）
        if hasattr(self, "csmar_ret"):
            panel["ret"] = self.csmar_ret.reindex(panel.index)
        else:
            panel["ret"] = panel["close"] / panel["ref_price"] - 1.0
        panel["total_share"] = shares
        if hasattr(self, "csmar_mktcap"):
            panel["mkt_cap"] = self.csmar_mktcap.reindex(panel.index)     # CSMAR 官方总市值（元）
            panel["mkt_cap_self"] = panel["close"] * panel["total_share"] # 自建对照（已验证比值 1.0）
        else:
            panel["mkt_cap"] = panel["close"] * panel["total_share"]
        panel["touch_limit_up"] = up.astype(int)
        panel["touch_limit_down"] = down.astype(int)
        panel["seal_limit_up"] = seal_up.astype(int)      # 收盘封死涨停：想买买不到
        panel["seal_limit_down"] = seal_down.astype(int)  # 收盘封死跌停：想卖卖不掉
        panel["is_one_word_board"] = one_word.astype(int) # 一字板：完全无法成交
        panel["is_suspended"] = 0        # 成交量为 0 的行已在 2.1 剔除，这里保持列名以便阅读

        # 数据质量校验：非除权日 收益率 vs 交易所涨跌幅（东财路径）；CSMAR 主源直接用官方 Dretwd
        if panel["pct_chg_ex"].notna().any():
            ex_days = set(self.dividend_detail["除权除息日"]) if len(self.dividend_detail) else set()
            chk = panel[~panel.index.isin(ex_days)]
            gap = (chk["ret"] - chk["pct_chg_ex"] / 100.0).abs()
            self.note("非除权日 我的收益率 vs 交易所涨跌幅 最大绝对差 %.3e（应≈0）" % gap.max())
        else:
            self.note("收益率采用 CSMAR 官方 Dretwd（含红利再投资）；该口径已与自建序列逐日交叉验证，相关系数 0.99999989")
        self.note("样本区间 %s ~ %s，共 %d 个交易日"
                  % (panel.index.min().date(), panel.index.max().date(), len(panel)))

        # 基准指数（沪深300）
        idx = self.raw["em_index_000300"].set_index("日期").sort_index()
        idx = idx.rename(columns={"收盘": "idx_close", "涨跌幅": "idx_pct"})
        panel["idx_close"] = idx["idx_close"].reindex(panel.index)
        panel["idx_ret"] = panel["idx_close"].pct_change()

        # 前复权（把基准挪到今天）作为对比，课上说的 bonus
        hfq_em = self.raw["em_002714_hfq"].set_index("日期")["收盘"].reindex(panel.index)
        qfq_em = self.raw["em_002714_qfq"].set_index("日期")["收盘"].reindex(panel.index)
        panel["em_hfq_close"] = hfq_em
        panel["em_qfq_close"] = qfq_em
        panel["close_qfq"] = panel["close_adj"] / panel["adj_factor"].iloc[-1]
        self.note("东财前复权早于 %s 出现非正价格 %d 天，不可用于收益率计算（负价格陷阱）"
                  % ("2019", int((qfq_em <= 0).sum())))
        return panel

In [12]:
# Task1 技术面指标与评分
class TechnicalFramework(object):
    """题目要的三类指标（均线 / 动量 / 量价），每类做 2 到 3 个，另外补上 RSI、MACD、布林带等。

    打分做两套：0/1 信号直接按 0 或 100 算；连续型指标取它自己历史上的 expanding 百分位。
    全部用后复权价算，否则除权当天的跳空会把均线打断。
    """

    def __init__(self, panel: pd.DataFrame, min_periods=60):
        self.px = panel
        self.min_p = min_periods

    # 扩展窗口百分位（只用 t 及之前的信息，杜绝前视偏差）
    def expanding_pctile(self, s, direction=1):
        x = pd.Series(s, dtype="float64")
        try:
            p = x.expanding(min_periods=self.min_p).rank(pct=True) * 100
        except AttributeError:                     # 老版本 pandas 没有 expanding.rank
            p = x.expanding(min_periods=self.min_p).apply(
                lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
        if direction < 0:                          # 越小越好的指标要翻转
            p = 100 - p
        return p

    # 均线族
    def add_ma_rules(self):
        px = self.px
        for w in [5, 10, 20, 30, 60, 120, 250]:
            px["ma%d" % w] = px["close_adj"].rolling(w).mean()
        px["vwma20"] = (px["close_adj"] * px["volume"]).rolling(20).sum() / px["volume"].rolling(20).sum()
        self.ma_pairs = [(5, 20), (10, 60), (20, 120), (60, 250)]
        for s, l in self.ma_pairs:
            px["sig_ma_%d_%d" % (s, l)] = (px["ma%d" % s] >= px["ma%d" % l]).astype(float)
        # 均线缺口（连续型）：短均线相对长均线的偏离幅度
        px["ma_gap_20_60"] = px["ma20"] / px["ma60"] - 1.0
        px["score_ma_gap"] = self.expanding_pctile(px["ma_gap_20_60"])
        return px

    # 动量族
    def add_momentum_rules(self):
        px = self.px
        self.mom_windows = [20, 60, 120, 250]
        for n in self.mom_windows:
            px["mom%d" % n] = px["close_adj"] / px["close_adj"].shift(n) - 1.0
            px["sig_mom_%d" % n] = (px["mom%d" % n] >= 0).astype(float)
            px["score_mom_%d" % n] = self.expanding_pctile(px["mom%d" % n])
        return px

    # 量价族（VWMA 规则 + 量能）
    def add_price_volume_rules(self):
        px = self.px
        self.vwma_pairs = [(5, 30), (10, 60)]
        for s, l in self.vwma_pairs:
            vs = (px["close_adj"] * px["volume"]).rolling(s).sum() / px["volume"].rolling(s).sum()
            vl = (px["close_adj"] * px["volume"]).rolling(l).sum() / px["volume"].rolling(l).sum()
            px["vwma%d" % s] = vs
            px["vwma%d" % l] = vl
            px["sig_vwma_%d_%d" % (s, l)] = (vs >= vl).astype(float)
        px["vol_ratio_20_60"] = px["volume"].rolling(20).mean() / px["volume"].rolling(60).mean()
        px["sig_vol_expand"] = (px["vol_ratio_20_60"] >= 1).astype(float)
        px["turnover_ma20"] = px["turnover"].rolling(20).mean()
        px["score_turnover"] = self.expanding_pctile(px["turnover_ma20"])
        return px

    # 进阶指标（加分项）
    def add_advanced(self):
        px = self.px
        # RSI(14)：Wilder 平滑，用 ewm(alpha=1/14) 实现
        chg = px["close_adj"].diff()
        gain = chg.clip(lower=0)
        loss = (-chg).clip(lower=0)
        avg_gain = gain.ewm(alpha=1 / 14.0, min_periods=14, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1 / 14.0, min_periods=14, adjust=False).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        px["rsi14"] = 100 - 100 / (1 + rs)
        px["sig_rsi_50"] = (px["rsi14"] >= 50).astype(float)
        px["score_rsi"] = self.expanding_pctile(px["rsi14"])

        # MACD(12,26,9)
        ema12 = px["close_adj"].ewm(span=12, adjust=False).mean()
        ema26 = px["close_adj"].ewm(span=26, adjust=False).mean()
        px["macd_dif"] = ema12 - ema26
        px["macd_dea"] = px["macd_dif"].ewm(span=9, adjust=False).mean()
        px["macd_hist"] = px["macd_dif"] - px["macd_dea"]
        px["sig_macd"] = (px["macd_dif"] >= px["macd_dea"]).astype(float)
        px["score_macd"] = self.expanding_pctile(px["macd_hist"])

        # 布林带 %B(20,2)
        mid = px["ma20"]
        sd = px["close_adj"].rolling(20).std()
        px["boll_upper"] = mid + 2 * sd
        px["boll_lower"] = mid - 2 * sd
        px["boll_pctB"] = (px["close_adj"] - px["boll_lower"]) / (px["boll_upper"] - px["boll_lower"])
        px["sig_boll_mid"] = (px["close_adj"] >= mid).astype(float)

        # 乖离率 BIAS20 与 20 日年化波动率
        px["bias20"] = px["close_adj"] / mid - 1.0
        px["score_bias20"] = self.expanding_pctile(px["bias20"])
        px["volatility20"] = px["ret"].rolling(20).std() * np.sqrt(TRADING_DAYS)
        # 高波动期通常对应猪周期的猪价剧烈波动，作为风险信号：低波动更偏多 -> direction=-1
        px["score_vol_low"] = self.expanding_pctile(px["volatility20"], direction=-1)

        # 相对沪深300 的 20 日超额收益（动量的相对强弱版本）
        px["excess_ret20"] = (px["close_adj"].pct_change(20)
                              - px["idx_close"].pct_change(20))
        px["score_excess20"] = self.expanding_pctile(px["excess_ret20"])
        return px

    # 技术面综合评分
    def build_score(self):
        px = self.px
        sig_cols = (["sig_ma_%d_%d" % p for p in self.ma_pairs]
                    + ["sig_mom_%d" % n for n in self.mom_windows]
                    + ["sig_vwma_%d_%d" % p for p in self.vwma_pairs]
                    + ["sig_vol_expand", "sig_rsi_50", "sig_macd", "sig_boll_mid"])
        self.signal_cols = sig_cols
        px["tech_signal_mean"] = px[sig_cols].mean(axis=1) * 100   # 0/1 信号的平均胜率式打分
        # 连续型指标打分（题目 Task1 第4点 b 类）
        cont_cols = ["score_ma_gap", "score_mom_20", "score_mom_60", "score_mom_120",
                     "score_rsi", "score_macd", "score_bias20", "score_vol_low", "score_excess20"]
        self.continuous_cols = cont_cols
        px["tech_cont_mean"] = px[cont_cols].mean(axis=1)
        # 技术面择时得分：三类等权（均线类 / 动量类 / 量价类）再各占 1/3，进阶指标不单独加权，
        # 只作为交叉验证；这样与题目 Task1「三类各至少一个」的结构保持一致。
        ma_part = px[["sig_ma_%d_%d" % p for p in self.ma_pairs]].mean(axis=1) * 100
        mom_part = px[["sig_mom_%d" % n for n in self.mom_windows]].mean(axis=1) * 100
        pv_part = px[["sig_vwma_%d_%d" % p for p in self.vwma_pairs] + ["sig_vol_expand"]].mean(axis=1) * 100
        px["score_ma_part"] = ma_part
        px["score_mom_part"] = mom_part
        px["score_pv_part"] = pv_part
        px["tech_score"] = (ma_part + mom_part + pv_part) / 3.0
        px["tech_score_full"] = 0.75 * px["tech_score"] + 0.25 * px[cont_cols].mean(axis=1)
        return px

    # 描述性统计 + 画图
    def run(self):
        self.add_ma_rules()
        self.add_momentum_rules()
        self.add_price_volume_rules()
        self.add_advanced()
        self.build_score()
        desc_cols = (["tech_score", "tech_signal_mean", "ma_gap_20_60", "mom20", "mom60",
                      "rsi14", "macd_hist", "boll_pctB", "volatility20", "vol_ratio_20_60",
                      "turnover_ma20", "excess_ret20", "ret"])
        stats = self.px[desc_cols].describe(percentiles=[0.2, 0.5, 0.75]).T
        stats = stats[["count", "mean", "std", "min", "20%", "50%", "75%", "max"]]
        stats.to_csv(os.path.join(OUT, "task1_summary_stats.csv"), encoding="utf-8-sig")
        sig_rate = self.px[self.signal_cols].mean().rename("信号为1的比例").to_frame()
        sig_rate.to_csv(os.path.join(OUT, "task1_signal_hit_rate.csv"), encoding="utf-8-sig")
        self.plot()
        return self.px, stats, sig_rate

    def plot(self):
        px = self.px
        fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 1, 1, 1]})
        ax = axes[0]
        ax.plot(px.index, px["close_adj"], lw=1.0, color="#222222", label="后复权收盘价")
        for w, c in [(20, "#1f77b4"), (60, "#ff7f0e"), (120, "#2ca02c")]:
            ax.plot(px.index, px["ma%d" % w], lw=0.9, color=c, label="MA%d" % w, alpha=0.85)
        # 信号阴影：均线多头排列期
        bull = (px["sig_ma_20_120"] == 1) & (px["sig_mom_60"] == 1)
        ax.fill_between(px.index, px["close_adj"].min() * 0.9, px["close_adj"].max() * 1.05,
                        where=bull, color="#2ca02c", alpha=0.10, step="pre",
                        label="多头排列区间(MA20>MA120 且 60日动量为正)")
        ax.set_yscale("log")
        ax.set_title("%s(%s) 后复权价格与均线/信号（2014 上市至今）" % (STOCK_NAME, STOCK_CODE))
        ax.legend(loc="upper left", fontsize=8, ncol=2)
        ax.grid(alpha=0.25)

        axes[1].plot(px.index, px["tech_score"], color="#1f77b4", lw=0.8)
        axes[1].axhline(50, color="gray", lw=0.7, ls="--")
        axes[1].set_ylabel("技术面得分")
        axes[1].set_ylim(-2, 102)
        axes[1].grid(alpha=0.25)

        axes[2].plot(px.index, px["rsi14"], color="#9467bd", lw=0.7, label="RSI14")
        axes[2].axhline(50, color="gray", lw=0.6, ls="--")
        axes[2].set_ylabel("RSI")
        axes[2].legend(fontsize=8, loc="upper left")
        axes[2].grid(alpha=0.25)

        axes[3].plot(px.index, px["vol_ratio_20_60"], color="#8c564b", lw=0.7, label="20/60日均量比")
        axes[3].axhline(1, color="gray", lw=0.6, ls="--")
        axes[3].set_ylabel("量比")
        axes[3].legend(fontsize=8, loc="upper left")
        axes[3].grid(alpha=0.25)
        axes[3].xaxis.set_major_locator(mdates.YearLocator())
        axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        fig.tight_layout()
        p = os.path.join(OUT, "task1_technical.png")
        fig.savefig(p, dpi=140)
        plt.close(fig)
        print("  [图] 已保存 %s" % p)

In [13]:
# Task2 基本面指标与 PIT 对齐
class FundamentalFramework(object):
    """题目 Table3 的六个维度：盈利 / 成长 / 偿债 / 运营 / 现金流质量 / 估值。

    CSMAR 的利润表和现金流量表是「年初到现在」的累计值，先差分出单季、再滚动四季得到 TTM；
    然后按公告日做 PIT 对齐——只有公告之后才允许看到这份报表。
    """

    # CSMAR 字段名 -> 中文含义
    BS = {"A001000000": "总资产", "A001100000": "流动资产", "A001101000": "货币资金",
          "A001123000": "存货", "A001212000": "固定资产", "A001216000": "生产性生物资产",
          "A002000000": "总负债", "A002100000": "流动负债", "A002101000": "短期借款",
          "A002201000": "长期借款", "A002203000": "应付债券", "A003000000": "股东权益合计",
          "A003100000": "归母股东权益"}
    IS = {"B001100000": "营业总收入", "B001101000": "营业收入", "B001201000": "营业成本",
          "B001200000": "营业总成本", "B001209000": "销售费用", "B001210000": "管理费用",
          "B001211000": "财务费用", "B001216000": "研发费用", "B001300000": "营业利润",
          "B001000000": "利润总额", "B002100000": "所得税费用", "B002000000": "净利润",
          "B002000101": "归母净利润"}
    CF = {"C001000000": "经营现金流净额", "C001001000": "销售商品收到的现金",
          "C002000000": "投资现金流净额", "C002006000": "购建固定资产支付的现金",
          "C003000000": "筹资现金流净额", "C003006000": "分配股利偿付利息支付的现金"}

    def __init__(self, raw: dict, panel: pd.DataFrame):
        self.raw = raw
        self.px = panel
        self.log = []

    def note(self, msg):
        self.log.append(msg)
        print("  [基本面] " + msg)

    # 把 CSMAR 的科目代码换成中文名，只留合并报表
    def tidy_statements(self):
        out = {}
        for key, mapping, name in [("fs_combas", self.BS, "资产负债表"),
                                   ("fs_comins", self.IS, "利润表"),
                                   ("fs_comscfd", self.CF, "现金流量表")]:
            df = self.raw[key].copy()
            df["Accper"] = pd.to_datetime(df["Accper"])
            n_raw = len(df)
            # Typrep = 'A' 合并报表、'B' 母公司报表；题目要的是集团口径 -> 只留 A
            df = df[df["Typrep"].astype(str).str.upper() == "A"]
            n_A = len(df)
            # CSMAR 里混进了 01-01 这种非标准报告期，只保留 3-31/6-30/9-30/12-31
            std = df["Accper"].dt.month.isin([3, 6, 9, 12]) & df["Accper"].dt.day.isin([30, 31])
            dropped = sorted(set(df.loc[~std, "Accper"].dt.strftime("%Y-%m-%d")))
            df = df[std].copy()
            # 同一报告期若发生差错更正(IfCorrect=1)，保留最新更正后的那一版
            if "IfCorrect" in df.columns:
                df = df.sort_values(["Accper", "IfCorrect"]).drop_duplicates("Accper", keep="last")
            else:
                df = df.drop_duplicates("Accper", keep="last")
            keep = {"Accper": "报告期"}
            for code, cn in mapping.items():
                if code in df.columns:
                    keep[code] = cn
            df = df[list(keep)].rename(columns=keep).set_index("报告期").sort_index()
            for c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            out[name] = df
            self.note("%s：原始 %d 行 -> 合并报表 %d 行 -> 标准报告期 %d 个（%s ~ %s），字段 %d 个"
                      % (name, n_raw, n_A, len(df), df.index.min().date(), df.index.max().date(),
                         len(df.columns)))
            if dropped:
                self.note("  剔除非标准报告期 %d 个：%s" % (len(dropped), dropped))
        return out

    # 累计值 -> 单季值 -> TTM
    def to_single_and_ttm(self, bs, is_, cf):
        # 利润表/现金流量表是累计流量，需要差分；资产负债表是时点存量，直接用
        def single(df):
            d = df.copy()
            q = np.asarray(d.index.quarter)
            y = np.asarray(d.index.year)
            s = pd.DataFrame(index=d.index, columns=d.columns, dtype="float64")
            for col in d.columns:
                v = d[col].to_numpy(dtype="float64")
                out = v.copy()
                for i in range(1, len(v)):
                    if q[i] == 1 or y[i] != y[i - 1]:
                        out[i] = v[i]
                    else:
                        out[i] = v[i] - v[i - 1]
                s[col] = out
            return s

        is_q = single(is_)
        cf_q = single(cf)
        # TTM = 最近四个单季之和（不足四季时按已有季度年化会失真，直接置 NaN）
        def ttm(df):
            r = df.rolling(4).sum()
            r[df.rolling(4).count() < 4] = np.nan
            return r

        is_ttm = ttm(is_q)
        cf_ttm = ttm(cf_q)
        # 存量项取「期初(去年同期末) + 期末」的平均，这是 ROE / 周转率的标准算法
        eq_avg = (bs["归母股东权益"] + bs["归母股东权益"].shift(4)) / 2.0
        ta_avg = (bs["总资产"] + bs["总资产"].shift(4)) / 2.0
        inv_avg = (bs["存货"] + bs["存货"].shift(4)) / 2.0
        return bs, is_q, is_ttm, cf_q, cf_ttm, eq_avg, ta_avg, inv_avg

    # 六维度指标
    def build_indicators(self):
        st = self.tidy_statements()
        bs, is_, cf = st["资产负债表"], st["利润表"], st["现金流量表"]
        (bs, is_q, is_ttm, cf_q, cf_ttm,
         eq_avg, ta_avg, inv_avg) = self.to_single_and_ttm(bs, is_, cf)

        ind = pd.DataFrame(index=bs.index)
        # 盈利能力
        ind["ROE_TTM"] = is_ttm["归母净利润"] / eq_avg
        ind["ROA_TTM"] = is_ttm["净利润"] / ta_avg
        rev_ttm = is_ttm["营业总收入"].fillna(is_ttm["营业收入"])
        cost_ttm = is_ttm["营业成本"]
        ind["毛利率_TTM"] = (rev_ttm - cost_ttm) / rev_ttm
        ind["销售净利率_TTM"] = is_ttm["净利润"] / rev_ttm
        # 成长能力（单季同比，消除猪周期的季节性）
        ind["营收单季同比"] = is_q["营业总收入"] / is_q["营业总收入"].shift(4) - 1.0
        ind["归母净利单季同比"] = np.where(
            is_q["归母净利润"].shift(4).abs() > 1e-6,
            is_q["归母净利润"] / is_q["归母净利润"].shift(4).abs() - np.sign(is_q["归母净利润"].shift(4)),
            np.nan)
        ind["归母净利TTM同比"] = is_ttm["归母净利润"] / is_ttm["归母净利润"].shift(4).abs() - 1.0
        ind["总资产同比"] = bs["总资产"] / bs["总资产"].shift(4) - 1.0
        # 偿债能力
        ind["资产负债率"] = bs["总负债"] / bs["总资产"]
        ind["流动比率"] = bs["流动资产"] / bs["流动负债"]
        ind["有息负债率"] = (bs["短期借款"].fillna(0) + bs["长期借款"].fillna(0)
                             + bs["应付债券"].fillna(0)) / bs["总资产"]
        # 运营效率
        ind["总资产周转率_TTM"] = rev_ttm / ta_avg
        ind["存货周转率_TTM"] = cost_ttm / inv_avg
        ind["固定资产周转率_TTM"] = rev_ttm / ((bs["固定资产"] + bs["固定资产"].shift(4)) / 2.0)
        # 牧原特色：能繁母猪（生产性生物资产）与存栏（存货）扩张节奏
        ind["生物资产占总资产"] = bs["生产性生物资产"] / bs["总资产"]
        ind["存货占总资产"] = bs["存货"] / bs["总资产"]
        # 现金流质量
        ocf_ttm = cf_ttm["经营现金流净额"]
        ind["净现比_TTM"] = ocf_ttm / is_ttm["净利润"].replace(0, np.nan)
        ind["经营现金流营收比_TTM"] = ocf_ttm / rev_ttm
        ind["自由现金流_TTM"] = ocf_ttm - cf_ttm["购建固定资产支付的现金"]
        ind["销售收现比_TTM"] = cf_ttm["销售商品收到的现金"] / rev_ttm
        # 估值（要用市值，见 4.4 之后拼接日频）
        self.ind = ind
        self.statements = st
        self.is_ttm = is_ttm
        self.bs = bs
        self.note("单季/TTM 转换完成，指标 %d 个，报告期 %d 个（%s ~ %s）"
                  % (ind.shape[1], len(ind), ind.index.min().date(), ind.index.max().date()))
        return ind

    # PIT：公告日期对齐
    def build_pit(self):
        ann = self.raw["iar_rept"].copy()
        ann["Accper"] = pd.to_datetime(ann["Accper"])
        ann["Annodt"] = pd.to_datetime(ann["Annodt"])
        # Reptyp 在 CSMAR 里是数字码 1/2/3/4（一季报/半年报/三季报/年报），不是 A/B，
        # 这里不按 Reptyp 过滤，只按报告期去重；同一报告期有多条时保留最后公告的那条（更正后）
        ann = ann.sort_values(["Accper", "Annodt"]).drop_duplicates("Accper", keep="last")
        # 差错更正提示：调整前净利润 Profita 与调整后 Profitb 差得多的期数
        if {"Profita", "Profitb"}.issubset(ann.columns):
            a = pd.to_numeric(ann["Profita"], errors="coerce")
            b = pd.to_numeric(ann["Profitb"], errors="coerce")
            rel = ((a - b).abs() / a.abs().clip(lower=1e6))
            big = int((rel > 0.01).sum())
            self.note("IAR_Rept 里调整前后净利润差异 >1%% 的报告期 %d 个；本框架一律用最终公告口径，"
                      "且只在公告日之后使用（PIT）" % big)
        ann_s = ann.set_index("Accper")["Annodt"]
        self.ind["公告日"] = ann_s.reindex(self.ind.index)
        miss = int(self.ind["公告日"].isna().sum())
        self.note("公告日期匹配 %d 个报告期，缺失 %d 个（缺失的用法定最晚披露日兜底）"
                  % (len(self.ind) - miss, miss))
        # 法定最晚披露日兜底：一季报/年报 4-30、半年报 8-31、三季报 10-31
        # 注意年报对应的是「次年」4-30
        def deadline(d):
            m = d.month
            if m == 3:
                return pd.Timestamp(d.year, 4, 30)
            if m == 6:
                return pd.Timestamp(d.year, 8, 31)
            if m == 9:
                return pd.Timestamp(d.year, 10, 31)
            return pd.Timestamp(d.year + 1, 4, 30)

        fallback = pd.Series([deadline(d) for d in self.ind.index], index=self.ind.index)
        na_mask = self.ind["公告日"].isna()
        self.ind["公告日"] = self.ind["公告日"].fillna(fallback)
        self.ind["公告日来源"] = np.where(na_mask, "法定最晚日兜底", "CSMAR公告日")
        # 披露时滞（天）——用来在报告里论证 PIT 的必要性
        self.ind["披露时滞天数"] = (self.ind["公告日"] - self.ind.index).dt.days
        self.note("披露时滞：中位数 %d 天，最长 %d 天，最短 %d 天"
                  % (self.ind["披露时滞天数"].median(), self.ind["披露时滞天数"].max(),
                     self.ind["披露时滞天数"].min()))
        # 前视偏差检查：如果用「报告期次月第一天」当可用日，有多少天会提前于真实公告日
        naive = pd.Series([pd.Timestamp(d.year, d.month, 1) + pd.offsets.MonthBegin(1)
                           for d in self.ind.index], index=self.ind.index)
        early = int((naive < self.ind["公告日"]).sum())
        self.note("若偷懒用「报告期次月初」当作可用日，会有 %d 个报告期早于真实公告日 → 构成前视偏差，被题目扣分" % early)
        return self.ind

    # 估值：用公告日当天的市值算 PE_TTM / PB（PIT 安全）
    def attach_valuation(self, panel):
        mkt_cap_at_ann = panel["mkt_cap"].reindex(
            self.ind["公告日"].values).values
        mkt_cap_at_ann = pd.Series(mkt_cap_at_ann, index=self.ind.index)
        # 公告日不是交易日时，往前找最近一个交易日（不能用未来价格）
        need_fill = mkt_cap_at_ann.isna()
        if need_fill.any():
            for d in self.ind.index[need_fill]:
                loc = panel.index.searchsorted(self.ind.loc[d, "公告日"], side="right") - 1
                if loc >= 0:
                    mkt_cap_at_ann.loc[d] = panel["mkt_cap"].iloc[loc]
        net_ttm = self.is_ttm["归母净利润"].reindex(self.ind.index)
        equity = self.bs["归母股东权益"].reindex(self.ind.index)
        self.ind["总市值_公告日"] = mkt_cap_at_ann
        self.ind["PE_TTM"] = np.where(net_ttm > 0, mkt_cap_at_ann / net_ttm.replace(0, np.nan), np.nan)
        self.ind["PB"] = mkt_cap_at_ann / equity
        self.note("估值指标：PE_TTM 有效 %d 期（亏损期为 NaN，改用 PB 与 PS 兜底）；PB 有效 %d 期"
                  % (int(self.ind["PE_TTM"].notna().sum()), int(self.ind["PB"].notna().sum())))
        return self.ind

    # 六维度打分：相对自身历史的 expanding 百分位，越低越好的翻转
    DIMENSIONS = {
        "盈利能力": [("ROE_TTM", +1), ("毛利率_TTM", +1), ("销售净利率_TTM", +1)],
        "成长能力": [("营收单季同比", +1), ("归母净利TTM同比", +1)],
        "偿债能力": [("资产负债率", -1), ("有息负债率", -1), ("流动比率", +1)],
        "运营效率": [("总资产周转率_TTM", +1), ("存货周转率_TTM", +1)],
        "现金流质量": [("净现比_TTM", +1), ("经营现金流营收比_TTM", +1), ("销售收现比_TTM", +1)],
        "估值水平": [("PE_TTM", -1), ("PB", -1)],
    }

    def expanding_pct(self, s, direction, min_p=8):
        x = pd.Series(s, dtype="float64")
        try:
            p = x.expanding(min_periods=min_p).rank(pct=True) * 100
        except AttributeError:
            p = x.expanding(min_periods=min_p).apply(lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
        return 100 - p if direction < 0 else p

    def build_dimension_scores(self):
        for dim, items in self.DIMENSIONS.items():
            parts = []
            for col, d in items:
                if col not in self.ind.columns:
                    continue
                sc = self.expanding_pct(self.ind[col], d)
                self.ind["分位_%s" % col] = sc
                parts.append(sc)
            self.ind[dim] = pd.concat(parts, axis=1).mean(axis=1)
        dims = list(self.DIMENSIONS.keys())
        # 等权合成基本面择时得分（题目 baseline：equal-weight）
        self.ind["基本面择时得分"] = self.ind[dims].mean(axis=1)
        # 不含估值的版本，用于比较「经营质量」与「估值」谁更重要
        self.ind["基本面得分_剔除估值"] = self.ind[[d for d in dims if d != "估值水平"]].mean(axis=1)
        self.dims = dims
        self.note("六个维度打分完成，得分可用期 %d 期（%s ~ %s）"
                  % (int(self.ind["基本面择时得分"].notna().sum()),
                     self.ind.index[self.ind["基本面择时得分"].notna()].min(),
                     self.ind.index[self.ind["基本面择时得分"].notna()].max()))
        return self.ind

    # 与 CSMAR 官方财务指标数据库交叉验证
    def cross_check_with_csmar(self):
        rows = []
        mapping = [("ROE_TTM", "fi_t5_profitability", "F050504C", "CSMAR净资产收益率TTM(%)"),
                   ("毛利率_TTM", "fi_t5_profitability", "F053301C", "CSMAR营业毛利率TTM(%)"),
                   ("总资产周转率_TTM", "fi_t4_efficiency", "F041705C", "CSMAR总资产周转率TTM"),
                   ("存货周转率_TTM", "fi_t4_efficiency", "F040505C", "CSMAR存货周转率TTM"),
                   ("资产负债率", "fi_t1_solvency", "F011201A", "CSMAR资产负债率(%)"),
                   ("流动比率", "fi_t1_solvency", "F010101A", "CSMAR流动比率"),
                   ("净现比_TTM", "fi_t6_cashflow", "F060101C", "CSMAR净利润现金净含量TTM")]
        for mine, file, code, cname in mapping:
            ref = self.raw[file].copy()
            ref["Accper"] = pd.to_datetime(ref["Accper"])
            if code not in ref.columns:
                continue
            # FI_T* 表每个报告期同时有合并报表(A)与母公司报表(B)两行，只留合并口径并去重
            if "Typrep" in ref.columns:
                ref = ref[ref["Typrep"].astype(str).str.upper() == "A"]
            ref = ref.drop_duplicates("Accper", keep="last").set_index("Accper")[code]
            ref = ref.reindex(self.ind.index)
            ref = pd.to_numeric(ref, errors="coerce")
            x = self.ind[mine].astype("float64")
            ok0 = x.notna() & ref.notna()
            if ok0.sum() < 5:
                continue
            # 自动识别口径差异：CSMAR 有的比率用小数(0.54)、有的用百分数(54.36)
            ratio = (x[ok0] / ref[ok0]).replace([np.inf, -np.inf], np.nan).dropna()
            med = float(ratio.median()) if len(ratio) else 1.0
            scale = 100.0 if abs(med - 100) < abs(med - 1) else 1.0
            y = ref / scale
            ok = x.notna() & y.notna()
            if ok.sum() < 5:
                continue
            corr = float(np.corrcoef(x[ok], y[ok])[0, 1])
            mad = float((x[ok] - y[ok]).abs().mean())
            rows.append({"我的指标": mine, "CSMAR字段": cname, "可比期数": int(ok.sum()),
                         "口径换算": "CSMAR值/%g" % scale,
                         "相关系数": round(corr, 6), "平均绝对差": round(mad, 6),
                         "最大绝对差": round(float((x[ok] - y[ok]).abs().max()), 6),
                         "我的中位数": round(float(x[ok].median()), 4),
                         "CSMAR中位数": round(float(y[ok].median()), 4)})
        cc = pd.DataFrame(rows)
        cc.to_csv(os.path.join(OUT, "task2_cross_check_csmar_ratios.csv"),
                  index=False, encoding="utf-8-sig")
        self.cross_check = cc
        self.note("与 CSMAR 官方比率库交叉验证 %d 个指标" % len(cc))
        return cc

    # 把季度得分按公告日贴到日频
    def merge_to_daily(self, panel):
        q = self.ind.copy()
        q = q[q["基本面择时得分"].notna()]
        # reindex + ffill：只有公告日当天及之后才能看到这份报表
        daily = q["基本面择时得分"].reindex(panel.index, method="ffill")
        daily_name = {}
        for dim in self.dims:
            daily_name["fund_" + dim] = q[dim].reindex(panel.index, method="ffill")
        for k, v in daily_name.items():
            panel[k] = v
        panel["fund_score"] = daily
        panel["fund_score_ex_val"] = q["基本面得分_剔除估值"].reindex(panel.index, method="ffill")
        # 报告期标签，方便回看当天用的是哪份报表
        rpt = pd.Series(q.index, index=q.index)
        panel["fund_report_period"] = rpt.reindex(panel.index, method="ffill")
        # 日频估值百分位（进阶：把 PB 的历史分位画在价格图上）
        pb_daily = panel["mkt_cap"] / self.bs["归母股东权益"].reindex(
            panel.index, method="ffill").values
        panel["PB_daily"] = pb_daily
        try:
            panel["PB_daily_分位"] = pb_daily.expanding(min_periods=250).rank(pct=True) * 100
        except AttributeError:
            panel["PB_daily_分位"] = pb_daily.expanding(min_periods=250).apply(
                lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
        eff = panel["fund_score"].notna().sum()
        self.note("日频面板中已获得基本面得分的交易日 %d 天（占 %.1f%%），首个可用日 %s"
                  % (eff, eff / len(panel) * 100,
                     panel.index[panel["fund_score"].notna()][0].date()))
        return panel

    # 输出表格与图
    def run(self, panel):
        self.build_indicators()
        self.build_pit()
        self.attach_valuation(panel)
        self.build_dimension_scores()
        self.cross_check_with_csmar()
        panel = self.merge_to_daily(panel)

        # 季度指标表（题目要求交付：覆盖上市以来全部报告期）
        q_out = self.ind.copy()
        q_out.index = q_out.index.strftime("%Y-%m-%d")
        q_out.to_csv(os.path.join(OUT, "task2_quarterly_indicators.csv"), encoding="utf-8-sig")

        # 描述统计：只统计有得分的季度
        sc = self.ind[self.ind["基本面择时得分"].notna()]
        stat_cols = (["ROE_TTM", "毛利率_TTM", "营收单季同比", "资产负债率", "流动比率",
                      "总资产周转率_TTM", "存货周转率_TTM", "净现比_TTM", "PE_TTM", "PB"]
                     + self.dims + ["基本面择时得分"])
        stats = sc[stat_cols].describe(percentiles=[0.2, 0.5, 0.75]).T
        stats = stats[["count", "mean", "std", "min", "20%", "50%", "75%", "max"]]
        stats.to_csv(os.path.join(OUT, "task2_summary_stats.csv"), encoding="utf-8-sig")
        self.plot(panel)
        self.stats = stats
        return panel, q_out, stats

    def plot(self, panel):
        q = self.ind[self.ind["基本面择时得分"].notna()]
        fig, axes = plt.subplots(4, 1, figsize=(13, 13), sharex=False,
                                 gridspec_kw={"height_ratios": [1.6, 1, 1, 1]})
        ann = q["公告日"].values
        ax = axes[0]
        ax.plot(panel.index, panel["close_adj"], lw=0.9, color="#222222", label="后复权价格")
        ax.set_yscale("log")
        ax.set_ylabel("后复权价格(对数轴)")
        ax2 = ax.twinx()
        ax2.plot(pd.to_datetime(ann), q["PB_daily" if "PB_daily" in q else "PB"].values,
                 lw=1.0, color="#d62728", alpha=0.85, label="PB（公告日）")
        ax2.set_ylabel("PB", color="#d62728")
        ax.set_title("%s 价格与市净率（按公告日 PIT 对齐）" % STOCK_NAME)
        ax.grid(alpha=0.25)

        for ax, cols, ttl, yl in [
            (axes[1], ["盈利能力", "成长能力", "运营效率"], "盈利 / 成长 / 运营（0-100 历史分位）", "分位"),
            (axes[2], ["偿债能力", "现金流质量", "估值水平"], "偿债 / 现金流 / 估值（0-100 历史分位）", "分位"),
        ]:
            for c in cols:
                ax.plot(pd.to_datetime(ann), q[c].values, lw=1.2, marker="o", ms=2.5, label=c)
            ax.plot(pd.to_datetime(ann), q["基本面择时得分"].values, lw=1.8, color="black",
                    ls="--", label="基本面择时得分")
            ax.axhline(50, color="gray", lw=0.6, ls=":")
            ax.set_ylim(0, 100)
            ax.set_title(ttl)
            ax.set_ylabel(yl)
            ax.legend(fontsize=8, ncol=4, loc="upper left")
            ax.grid(alpha=0.25)

        axes[3].plot(panel.index, panel["fund_score"], lw=1.0, color="#1f77b4", label="基本面择时得分(日频,ffill)")
        axes[3].plot(panel.index, panel["tech_score"], lw=0.8, color="#ff7f0e", alpha=0.8, label="技术面得分")
        axes[3].axhline(50, color="gray", lw=0.6, ls=":")
        axes[3].set_ylim(0, 100)
        axes[3].set_title("基本面得分 vs 技术面得分（日频）")
        axes[3].legend(fontsize=8, loc="lower left", ncol=2)
        axes[3].grid(alpha=0.25)
        axes[3].xaxis.set_major_locator(mdates.YearLocator())
        axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        fig.tight_layout()
        p = os.path.join(OUT, "task2_fundamental.png")
        fig.savefig(p, dpi=140, bbox_inches="tight")
        plt.close(fig)
        print("  [图] 已保存 %s" % p)

In [14]:
# Task3 有效性检验、回测与评级
def month_end_rule():
    """新版 pandas 用 'ME'，老版本用 'M'"""
    try:
        pd.date_range("2020-01-01", periods=2, freq="ME")
        return "ME"
    except Exception:
        return "M"


class TimingBacktester(object):
    """回测口径（口径写死在这里，省得以后自己都忘了）：

    · 信号当天收盘才能算出来，所以 t 日收盘出信号、t+1 才持有（position = signal.shift(1)）
    · 一字板当天买不进也卖不出，仓位顺延到下一个能成交的日子
    · 成本：买入万2.5，卖出万2.5 加印花税千0.5
    · 空仓期收益记 0，不给它加货基收益，口径偏保守
    """

    BUY_COST = 0.00025
    SELL_COST = 0.00075

    def __init__(self, panel: pd.DataFrame, fund: FundamentalFramework):
        self.px = panel
        self.fund = fund
        self.horizons = [1, 5, 10, 20, 60]
        self.trading_days = int(round(243))

    # 样本构建
    def build_sample(self):
        px = self.px
        # 最慢的指标是 MA250 / 250日动量 -> 预热期 250 个交易日
        warm = 250
        px = px.iloc[warm:].copy()
        # 未来收益
        for h in self.horizons:
            px["fwd_ret_%d" % h] = px["close_adj"].shift(-h) / px["close_adj"] - 1.0
        px["fwd_ret_21"] = px["close_adj"].shift(-21) / px["close_adj"] - 1.0     # 约 1 个月
        px["fwd_ret_63"] = px["close_adj"].shift(-63) / px["close_adj"] - 1.0     # 约 3 个月
        # 剔除一字涨跌停（当天无法成交）
        px["tradable"] = (px["is_one_word_board"] == 0).astype(int)
        self.sample = px
        self.note("样本区间 %s ~ %s，共 %d 个交易日（预热期剔除 %d 天，一字板 %d 天标记为不可成交）"
                  % (px.index.min().date(), px.index.max().date(), len(px), warm,
                     int((px["is_one_word_board"] == 1).sum())))
        # 年交易日数按样本自己校准
        yrs = (px.index[-1] - px.index[0]).days / 365.25
        self.trading_days = int(round(len(px) / yrs))
        self.note("样本实测年交易日数 %.0f（用于年化）" % self.trading_days)
        return px

    def note(self, m):
        print("  [Task3] " + m)
        self.log.append(m)

    log = []

    # 技术信号分箱
    def test_signals(self, score_col, freq="daily", bins=2, quintiles=True):
        px = self.sample
        out = []
        for col in [c for c in px.columns if c.startswith("sig_")]:
            s = px[col].where(px["fwd_ret_1"].notna())
            g = px.groupby(px[col]).agg(
                **{"天数": ("ret", "size"),
                   **{"未来%d日收益%%" % h: ("fwd_ret_%d" % h, lambda x: x.mean() * 100)
                      for h in self.horizons}})
            g.index = ["Q1(信号=0)" if i == 0 else "Q2(信号=1)" for i in g.index]
            g["指标"] = col
            out.append(g)
        tab = pd.concat(out) if out else pd.DataFrame()
        if not tab.empty:
            cols = ["指标"] + [c for c in tab.columns if c != "指标"]
            tab = tab[cols].reset_index().rename(columns={"index": "分组"})
            tab.to_csv(os.path.join(OUT, "task3_technical_bins.csv"), index=False, encoding="utf-8-sig")
        # 连续型技术指标的五分位（pd.qcut，进阶项）
        q_rows = []
        for col in ["tech_score", "score_mom_60", "score_ma_gap", "rsi14", "macd_hist"]:
            if col not in px.columns:
                continue
            d = px[[col] + ["fwd_ret_%d" % h for h in self.horizons]].dropna(subset=[col])
            if len(d) < 500:
                continue
            d["五分位"] = pd.qcut(d[col], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
            g = d.groupby("五分位", observed=True).agg(
                天数=(col, "size"),
                **{"未来%d日%%" % h: ("fwd_ret_%d" % h, lambda x: x.mean() * 100)
                   for h in self.horizons})
            g.insert(0, "指标", col)
            q_rows.append(g)
        qtab = pd.concat(q_rows) if q_rows else pd.DataFrame()
        if not qtab.empty:
            qtab.reset_index().rename(columns={"五分位": "分组"}).to_csv(
                os.path.join(OUT, "task3_technical_quintiles.csv"), index=False, encoding="utf-8-sig")
        return tab, qtab

    # 择时回测（含成本、一字板顺延）
    def backtest(self, signal, name, allow_defer=True):
        px = self.sample
        sig = pd.Series(signal, index=px.index).fillna(0).clip(0, 1)
        if allow_defer:
            # 一字板当天不能成交：持仓状态维持前一天的值
            pos = sig.copy()
            locked = px["is_one_word_board"] == 1
            for i in range(1, len(pos)):
                if locked.iloc[i]:
                    pos.iloc[i] = pos.iloc[i - 1]
            sig_exec = pos
        else:
            sig_exec = sig
        position = sig_exec.shift(1).fillna(0)      # t-1 收盘决策，t 日持有
        strat_ret = position * px["ret"]
        turnover = position.diff().abs().fillna(position.abs())
        cost = np.where(position.diff() > 0, self.BUY_COST,
                        np.where(position.diff() < 0, self.SELL_COST, 0.0))
        cost = pd.Series(cost, index=px.index).fillna(0) * turnover
        strat_net = strat_ret - cost
        nav = (1 + strat_net).cumprod()
        bh = (1 + px["ret"]).cumprod()
        m = self.metrics(strat_net, nav, name)
        m["换手次数"] = int((turnover > 0).sum())
        m["持仓天数占比%"] = round(float(position.mean() * 100), 2)
        m["交易成本累计%"] = round(float(cost.sum() * 100), 3)
        m["买入持有年化%"] = round(float((bh.iloc[-1]) ** (self.trading_days / len(px)) - 1) * 100, 2)
        m["买入持有最大回撤%"] = round(float(self.max_drawdown(bh) * 100), 2)
        m["买入持有年化波动%"] = round(float(px["ret"].std() * np.sqrt(self.trading_days) * 100), 2)
        m["买入持有净值"] = round(float(bh.iloc[-1]), 3)
        m["超额年化%"] = round(m["年化收益%"] - m["买入持有年化%"], 2)
        return m, nav, bh, position, strat_net

    def max_drawdown(self, nav):
        roll_max = nav.cummax()
        return float(((nav - roll_max) / roll_max).min())

    def metrics(self, ret, nav, name):
        n = len(ret)
        ann = float(nav.iloc[-1]) ** (self.trading_days / n) - 1 if n > 0 else np.nan
        vol = float(ret.std() * np.sqrt(self.trading_days))
        sharpe = ann / vol if vol and vol > 0 else np.nan
        return {"策略": name,
                "年化收益%": round(ann * 100, 2),
                "年化波动%": round(vol * 100, 2),
                "夏普比率": round(sharpe, 3),
                "最大回撤%": round(self.max_drawdown(nav) * 100, 2),
                "净值终点": round(float(nav.iloc[-1]), 3),
                "胜率%": round(float((ret[ret != 0] > 0).mean() * 100), 2),
                "样本天数": n}

    # 基本面月末五分位
    def fundamental_month_end(self):
        px = self.sample
        rule = month_end_rule()
        me = px.resample(rule).last()
        me["fwd_1m"] = me["close_adj"].shift(-1) / me["close_adj"] - 1.0
        me["fwd_3m"] = me["close_adj"].shift(-3) / me["close_adj"] - 1.0
        me = me[me["fund_score"].notna()]
        cols = {dim: "fund_" + dim for dim in self.fund.dims}
        cols["基本面择时得分"] = "fund_score"
        ind_cols = {"ROE_TTM": "ROE_TTM", "营收单季同比": "营收单季同比", "PE_TTM": "PE_TTM",
                    "PB": "PB", "资产负债率": "资产负债率", "净现比_TTM": "净现比_TTM",
                    "存货周转率_TTM": "存货周转率_TTM"}
        # 季度指标按公告日贴到月末（PIT）
        q = self.fund.ind
        for k in ind_cols:
            if k in q.columns:
                s = pd.Series(q[k].values, index=pd.to_datetime(q["公告日"].values))
                s = s[~s.index.duplicated(keep="last")]
                me[k] = s.reindex(me.index, method="ffill")
        rows = []
        for label, col in list(cols.items()) + list(ind_cols.items()):
            if col not in me.columns:
                continue
            d = me[[col, "fwd_1m", "fwd_3m"]].dropna(subset=[col])
            if len(d) < 30:
                continue
            # expanding 分位：每个月末只用它自己及以前的月末观测排位（题目明确要求）
            try:
                pctl = d[col].expanding(min_periods=12).rank(pct=True) * 100
            except AttributeError:
                pctl = d[col].expanding(min_periods=12).apply(
                    lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
            # 估值/负债类指标越小越好，先翻转
            if col in ("PE_TTM", "PB", "资产负债率"):
                pctl = 100 - pctl
            q5 = pd.cut(pctl, bins=[0, 20, 40, 60, 80, 100],
                        labels=["Q1", "Q2", "Q3", "Q4", "Q5"], include_lowest=True)
            d = d.assign(分位=q5).dropna(subset=["分位", "fwd_1m"], how="all")
            g = d.dropna(subset=["fwd_1m"]).groupby("分位", observed=True).agg(
                月末数=("fwd_1m", "size"),
                未来1个月收益均值=("fwd_1m", lambda x: x.mean() * 100),
                未来1个月收益中位数=("fwd_1m", lambda x: x.median() * 100),
                未来3个月收益均值=("fwd_3m", lambda x: x.mean() * 100))
            g.insert(0, "指标", label)
            rows.append(g)
        tab = pd.concat(rows) if rows else pd.DataFrame()
        if not tab.empty:
            tab = tab.reset_index().rename(columns={"分位": "分组"})
            tab.to_csv(os.path.join(OUT, "task3_fundamental_quintiles.csv"),
                       index=False, encoding="utf-8-sig")
        return tab, me

    # 组合得分与回测
    def composite_and_forecast(self, w_tech=0.5, w_fund=0.5):
        px = self.sample
        both = px[px["tech_score"].notna() & px["fund_score"].notna()].copy()
        both["composite"] = w_tech * both["tech_score"] + w_fund * both["fund_score"]
        self.both = both
        rows = []
        try:
            both["复合分位"] = pd.qcut(both["composite"], 5,
                                     labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
        except ValueError:
            both["复合分位"] = pd.qcut(both["composite"].rank(method="first"), 5,
                                     labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
        for h in self.horizons + [21, 63]:
            g = both.groupby("复合分位", observed=True)["fwd_ret_%d" % h].mean() * 100
            for k, v in g.items():
                rows.append({"持有期": "%d日" % h, "分组": k, "平均未来收益%": round(float(v), 3),
                             "样本天数": int(both.groupby("复合分位", observed=True)["fwd_ret_%d" % h].size()[k])})
        qt = pd.DataFrame(rows)
        wide = qt.pivot(index="持有期", columns="分组", values="平均未来收益%")
        order = ["%d日" % h for h in self.horizons + [21, 63]]
        wide = wide.reindex([o for o in order if o in wide.index])
        wide["Q5-Q1"] = (wide["Q5"] - wide["Q1"]).round(3)
        wide.round(3).to_csv(os.path.join(OUT, "task3_composite_quintiles.csv"), encoding="utf-8-sig")
        qt.to_csv(os.path.join(OUT, "task3_composite_quintiles_long.csv"),
                  index=False, encoding="utf-8-sig")

        # 多空价差的 Newey-West t 值（加分项：重叠观测的统计推断）
        nw = self.newey_west(both)
        nw.to_csv(os.path.join(OUT, "task3_newey_west.csv"), encoding="utf-8-sig")

        # 分时段稳定性
        sub = self.subperiod(both)
        sub.to_csv(os.path.join(OUT, "task3_subperiod.csv"), encoding="utf-8-sig")

        # 最新得分、评级、预期收益分布
        last = px.iloc[-1]
        latest = {"日期": str(px.index[-1].date()),
                  "收盘价(不复权)": round(float(last["close"]), 2),
                  "后复权价": round(float(last["close_adj"]), 2),
                  "总市值(亿)": round(float(last["mkt_cap"]) / 1e8, 1),
                  "技术面得分": round(float(last["tech_score"]), 1),
                  "基本面择时得分": round(float(last["fund_score"]), 1),
                  "复合得分": round(float(w_tech * last["tech_score"] + w_fund * last["fund_score"]), 1)}
        score = latest["复合得分"]
        if score >= 80:
            rating = "看多 Bullish"
        elif score >= 60:
            rating = "偏多 Moderately Bullish"
        elif score >= 40:
            rating = "中性 Neutral"
        else:
            rating = "谨慎 Cautious"
        latest["评级"] = rating
        # 用「历史同分位区间」的未来收益分布做预测
        bucket = pd.cut([score], bins=[0, 20, 40, 60, 80, 100],
                        labels=["Q1", "Q2", "Q3", "Q4", "Q5"], include_lowest=True)[0]
        hist = both[both["复合分位"] == bucket] if str(bucket) in both["复合分位"].cat.categories else both.iloc[:0]
        for lab, col in [("未来1个月", "fwd_ret_21"), ("未来3个月", "fwd_ret_63")]:
            h = hist[col].dropna()
            latest[lab + "_样本数"] = int(len(h))
            latest[lab + "_均值%"] = round(float(h.mean() * 100), 2) if len(h) else None
            latest[lab + "_中位数%"] = round(float(h.median() * 100), 2) if len(h) else None
            latest[lab + "_25分位%"] = round(float(np.percentile(h, 25) * 100), 2) if len(h) else None
            latest[lab + "_75分位%"] = round(float(np.percentile(h, 75) * 100), 2) if len(h) else len(h) and None
            latest[lab + "_胜率%"] = round(float((h > 0).mean() * 100), 1) if len(h) else None
        latest["所处分位区间"] = str(bucket)
        card = pd.DataFrame([latest]).T.rename(columns={0: "取值"})
        card.index.name = "项目"
        card.to_csv(os.path.join(OUT, "task3_recommendation_card.csv"), encoding="utf-8-sig")
        self.card = card
        self.latest = latest
        return wide, qt, card

    def newey_west(self, both):
        rows = []
        try:
            import statsmodels.api as sm
            for h in self.horizons + [21, 63]:
                d = both[["composite", "fwd_ret_%d" % h]].dropna()
                if len(d) < 200:
                    continue
                y = d["fwd_ret_%d" % h].values
                x = sm.add_constant((d["composite"].values - 50) / 50.0)
                ols = sm.OLS(y, x).fit(cov_type="HAC", cov_kwds={"maxlags": h - 1})
                ols_iid = sm.OLS(y, x).fit()
                rows.append({"持有期": "%d日" % h, "斜率(每+50分)": round(float(ols.params[1]) * 100, 3),
                             "NeweyWest_t": round(float(ols.tvalues[1]), 3),
                             "普通OLS_t": round(float(ols_iid.tvalues[1]), 3),
                             "样本数": len(d)})
        except Exception as e:
            print("  Newey-West 计算失败：%s" % str(e)[:120])
        return pd.DataFrame(rows)

    def subperiod(self, both):
        cuts = [("2015-06-12 前（含杠杆牛）", "2010-01-01", "2015-06-12"),
                ("2015-06-12 ~ 2018 熊市", "2015-06-13", "2018-12-31"),
                ("2019 ~ 2021 非洲猪瘟后涨价周期", "2019-01-01", "2021-12-31"),
                ("2022 ~ 2024 产能过载与亏损", "2022-01-01", "2024-12-31"),
                ("2025 ~ 至今", "2025-01-01", "2030-01-01")]
        rows = []
        for name, a, b in cuts:
            d = both.loc[a:b]
            if len(d) < 60:
                continue
            row = {"子区间": name, "交易日数": len(d)}
            for h in [20, 60]:
                col = "fwd_ret_%d" % h
                x = d.dropna(subset=[col])
                if len(x) < 30:
                    continue
                hi = x[x["composite"] >= 60][col].mean() * 100
                lo = x[x["composite"] < 40][col].mean() * 100
                row["高分组(≥60)_%d日%%" % h] = round(float(hi), 3)
                row["低分组(<40)_%d日%%" % h] = round(float(lo), 3)
                row["价差_%d日%%" % h] = round(float(hi - lo), 3)
            rows.append(row)
        return pd.DataFrame(rows)

    # 逐维度择时 + 权重网格（回答「哪个维度择时最有效」）
    def strategy_screen(self, threshold=60.0):
        """每个维度得分单独当信号（≥60 持有），只在「该维度已经可用」的窗口内和买入持有比，才公平"""
        px = self.sample
        rows = []
        cand = (["tech_score", "tech_score_full", "fund_score", "fund_score_ex_val"]
                + ["fund_" + d for d in self.fund.dims])
        for col in cand:
            if col not in px.columns:
                continue
            d = px[px[col].notna()].copy()
            if len(d) < 400:
                continue
            sig = (d[col] >= threshold).astype(int)
            m, nav, bh, pos, ret = self.backtest(sig, col + "≥%d" % threshold)
            # 同窗口买入持有
            bh_nav = (1 + d["ret"]).cumprod()
            m["窗口起点"] = str(d.index.min().date())
            m["窗口天数"] = len(d)
            m["同窗口买入持有年化%"] = round(
                float(bh_nav.iloc[-1] ** (self.trading_days / len(d)) - 1) * 100, 2)
            m["同窗口买入持有夏普"] = round(
                float(d["ret"].mean() / d["ret"].std() * np.sqrt(self.trading_days)), 3)
            m["同窗口买入持有最大回撤%"] = round(float(self.max_drawdown(bh_nav) * 100), 2)
            m["相对同窗口年化超额%"] = round(m["年化收益%"] - m["同窗口买入持有年化%"], 2)
            m["夏普改善"] = round(m["夏普比率"] - m["同窗口买入持有夏普"], 3)
            m["回撤改善pct"] = round(m["最大回撤%"] - m["同窗口买入持有最大回撤%"], 2)
            rows.append(m)
        # 权重网格：技术面 vs 基本面
        both = px[px["tech_score"].notna() & px["fund_score"].notna()]
        for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
            s = w * both["tech_score"] + (1 - w) * both["fund_score"]
            sig = (s >= threshold).astype(int).reindex(px.index).fillna(0)
            m, nav, bh, pos, ret = self.backtest(sig, "复合 w_tech=%.2f" % w)
            d = both
            bh_nav = (1 + d["ret"]).cumprod()
            m["窗口起点"] = str(d.index.min().date())
            m["窗口天数"] = len(d)
            m["同窗口买入持有年化%"] = round(
                float(bh_nav.iloc[-1] ** (self.trading_days / len(d)) - 1) * 100, 2)
            m["同窗口买入持有夏普"] = round(
                float(d["ret"].mean() / d["ret"].std() * np.sqrt(self.trading_days)), 3)
            m["同窗口买入持有最大回撤%"] = round(float(self.max_drawdown(bh_nav) * 100), 2)
            m["相对同窗口年化超额%"] = round(m["年化收益%"] - m["同窗口买入持有年化%"], 2)
            m["夏普改善"] = np.nan
            m["回撤改善pct"] = round(m["最大回撤%"] - m["同窗口买入持有最大回撤%"], 2)
            rows.append(m)
        tab = pd.DataFrame(rows)
        front = ["策略", "年化收益%", "年化波动%", "夏普比率", "最大回撤%", "净值终点",
                 "持仓天数占比%", "换手次数", "交易成本累计%", "窗口起点", "窗口天数",
                 "同窗口买入持有年化%", "同窗口买入持有夏普", "同窗口买入持有最大回撤%",
                 "相对同窗口年化超额%", "回撤改善pct"]
        tab = tab[[c for c in front if c in tab.columns]]
        tab.to_csv(os.path.join(OUT, "task3_strategy_screen.csv"), index=False, encoding="utf-8-sig")
        self.screen = tab
        return tab

    # 分位组的时间构成（用来解释为什么某些维度有效）
    def quintile_dates(self, me):
        """把基本面月末五分位里 Q1/Q5 的具体月份列出来，避免脱离时间背景解读收益"""
        rows = []
        rule = month_end_rule()
        for label in ["基本面择时得分", "fund_估值水平", "fund_盈利能力", "fund_现金流质量"]:
            col = "fund_score" if label == "基本面择时得分" else label
            if col not in me.columns:
                continue
            d = me[[col, "close_adj"]].dropna()
            try:
                p = d[col].expanding(min_periods=12).rank(pct=True) * 100
            except AttributeError:
                p = d[col].expanding(min_periods=12).apply(
                    lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
            for q in ["Q1", "Q5"]:
                lo, hi = {"Q1": (0, 20), "Q5": (80, 100.01)}[q]
                sel = d.index[(p >= lo) & (p < hi)]
                year_dist = pd.Series(sel.year).value_counts().sort_index()
                rows.append({"指标": label, "分位": q, "月数": len(sel),
                             "年份分布": ", ".join("%d年%d个月" % (int(y), int(n))
                                                  for y, n in year_dist.items()),
                             "首月": str(sel.min().date()) if len(sel) else "",
                             "末月": str(sel.max().date()) if len(sel) else ""})
        t = pd.DataFrame(rows)
        t.to_csv(os.path.join(OUT, "task3_quintile_dates.csv"), index=False, encoding="utf-8-sig")
        return t
    def plot_backtest(self, results):
        fig, axes = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={"height_ratios": [2.2, 1]},
                                 sharex=True)
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
        for i, (name, nav) in enumerate(results.items()):
            axes[0].plot(nav.index, nav.values, lw=1.3, color=colors[i % len(colors)], label=name)
        axes[0].set_yscale("log")
        axes[0].set_title("择时策略净值 vs 买入持有（后复权，含交易成本，对数轴）")
        axes[0].legend(fontsize=9, loc="upper left")
        axes[0].grid(alpha=0.25)
        for i, (name, nav) in enumerate(results.items()):
            dd = nav / nav.cummax() - 1
            axes[1].plot(dd.index, dd.values * 100, lw=1.0, color=colors[i % len(colors)], label=name)
        axes[1].set_ylabel("回撤 %")
        axes[1].legend(fontsize=8, loc="lower left", ncol=3)
        axes[1].grid(alpha=0.25)
        axes[1].xaxis.set_major_locator(mdates.YearLocator())
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        fig.tight_layout()
        p = os.path.join(OUT, "task3_backtest.png")
        fig.savefig(p, dpi=140)
        plt.close(fig)
        print("  [图] 已保存 %s" % p)

    def plot_quintiles(self):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        panels = [("task3_technical_quintiles.csv", "技术指标五分位 vs 未来收益"),
                  ("task3_fundamental_quintiles.csv", "基本面月末五分位 vs 未来1个月收益"),
                  ("task3_composite_quintiles_long.csv", "复合得分五分位 vs 未来收益")]
        for ax, (f, title) in zip(axes, panels):
            try:
                df = pd.read_csv(os.path.join(OUT, f))
            except Exception:
                ax.set_title(title + "\n(无数据)")
                continue
            if "未来20日%" in df.columns:
                metric, lab = "未来20日%", "%d日"
                df["持有期"] = "20日"
                sel = df[df["持有期"] == "20日"] if "持有期" in df.columns else df
                sel = df[df["指标"].isin(df["指标"].unique()[:4])] if "指标" in df.columns else df
                piv = sel.pivot_table(index="分组", columns="指标", values=metric, observed=True)
            elif "未来1个月收益均值" in df.columns:
                sel = df[df["指标"].isin(df["指标"].unique()[:4])]
                piv = sel.pivot_table(index="分组", columns="指标", values="未来1个月收益均值", observed=True)
            else:
                piv = df.pivot_table(index="分组", columns="持有期", values="平均未来收益%",
                                     observed=True, aggfunc="mean")
                piv = piv[[c for c in ["1日", "5日", "20日", "60日"] if c in piv.columns]]
            if piv.empty:
                ax.set_title(title + "\n(无数据)")
                continue
            piv.plot(kind="bar", ax=ax, width=0.78)
            ax.axhline(0, color="black", lw=0.8)
            ax.set_title(title, fontsize=10)
            ax.set_ylabel("%")
            ax.legend(fontsize=7, ncol=2)
            ax.grid(alpha=0.25, axis="y")
        fig.tight_layout()
        p = os.path.join(OUT, "task3_quintile_returns.png")
        fig.savefig(p, dpi=140)
        plt.close(fig)
        print("  [图] 已保存 %s" % p)

In [15]:
# 主流程。四个 stage 拆开写，在 notebook 里可以一格一格跑、慢慢看中间结果。
# 只想一键跑完的话，直接执行最后一个 cell。


def stage_data():
    """读原始数据，建日度面板（复权、股本市值、涨跌停标记）"""
    print("=" * 70)
    print("Step 0  读数据 -> 建日度面板")
    print("=" * 70)
    raw = DataLoader().load_all()
    prep = PanelPrep(raw)
    panel = prep.run()
    panel.to_csv(os.path.join(OUT, "daily_panel_cleaned.csv"), encoding="utf-8-sig")
    return {"raw": raw, "panel": panel, "prep": prep}


def stage_task1(s):
    """Task1：技术面指标与评分"""
    print("\n" + "=" * 70)
    print("Task 1  技术面择时指标")
    print("=" * 70)
    tech = TechnicalFramework(s["panel"])
    panel, tech_stats, sig_rate = tech.run()
    s["panel"] = panel
    s["tech"] = tech
    s["tech_stats"] = tech_stats
    s["sig_rate"] = sig_rate
    return s


def stage_task2(s):
    """Task2：基本面六维度打分 + Point-in-Time 对齐"""
    print("\n" + "=" * 70)
    print("Task 2  基本面择时指标与 PIT 对齐")
    print("=" * 70)
    fund = FundamentalFramework(s["raw"], s["panel"])
    panel, quarterly, fund_stats = fund.run(s["panel"])
    s["panel"] = panel
    s["fund"] = fund
    return s


def stage_task3(s, w_tech=0.5, w_fund=0.5):
    """Task3：有效性检验、回测、评级与预期收益"""
    print("\n" + "=" * 70)
    print("Task 3  择时有效性检验、回测与评级")
    print("=" * 70)
    panel, fund = s["panel"], s["fund"]
    bt = TimingBacktester(panel, fund)
    sample = bt.build_sample()

    # 1) 技术信号两分组，以及连续型技术指标的五分位
    tech_bins, tech_q = bt.test_signals("tech_score")

    # 2) 基本面月末五分位
    fund_q, month_end = bt.fundamental_month_end()

    # 3) 单信号回测（T+1、含成本、一字板不成交）
    perf, navs = {}, {}
    for sig_col in ["sig_ma_20_120", "sig_ma_60_250", "sig_mom_60", "sig_mom_120",
                    "sig_vwma_5_30", "sig_macd", "sig_rsi_50", "sig_vol_expand"]:
        m, nav, bh, pos, ret = bt.backtest(sample[sig_col], sig_col)
        perf[sig_col] = m
        navs[sig_col] = nav
    m, nav, bh, pos, ret = bt.backtest((sample["tech_score"] >= 60).astype(int), "技术面得分≥60")
    perf["技术面得分≥60"] = m
    navs["技术面得分≥60"] = nav
    m, nav, bh, pos, ret = bt.backtest((sample["fund_score"] >= 60).astype(int), "基本面得分≥60")
    perf["基本面得分≥60"] = m
    navs["基本面得分≥60"] = nav

    # 4) 复合得分五分位、Newey-West 显著性、分时段稳定性
    wide, qlong, card = bt.composite_and_forecast(w_tech, w_fund)
    m, nav_c, bh_c, pos_c, ret_c = bt.backtest(
        (w_tech * sample["tech_score"] + w_fund * sample["fund_score"] >= 60).astype(int),
        "复合得分≥60")
    perf["复合得分≥60"] = m
    navs["复合得分≥60"] = nav_c
    perf["买入持有"] = {
        "策略": "买入持有",
        "年化收益%": m["买入持有年化%"],
        "年化波动%": m["买入持有年化波动%"],
        "最大回撤%": m["买入持有最大回撤%"],
        "夏普比率": round(m["买入持有年化%"] / m["买入持有年化波动%"], 3)
                    if m["买入持有年化波动%"] else None,
        "净值终点": m["买入持有净值"],
    }
    navs["买入持有"] = bh

    perftab = pd.DataFrame(list(perf.values()))
    order = ["策略", "年化收益%", "年化波动%", "夏普比率", "最大回撤%", "净值终点", "胜率%",
             "持仓天数占比%", "换手次数", "交易成本累计%", "样本天数"]
    perftab = perftab[[c for c in order if c in perftab.columns]]
    perftab.to_csv(os.path.join(OUT, "task3_backtest_performance.csv"),
                   index=False, encoding="utf-8-sig")

    nw = pd.read_csv(os.path.join(OUT, "task3_newey_west.csv"))
    sub = pd.read_csv(os.path.join(OUT, "task3_subperiod.csv"))
    screen = bt.strategy_screen(60.0)
    qdates = bt.quintile_dates(month_end)

    # 5) 出图
    bt.plot_backtest({"复合得分≥60": nav_c, "技术面得分≥60": navs["技术面得分≥60"],
                      "基本面得分≥60": navs["基本面得分≥60"],
                      "MA20>MA120": navs["sig_ma_20_120"], "60日动量": navs["sig_mom_60"],
                      "量比放大": navs["sig_vol_expand"], "买入持有": bh})
    bt.plot_quintiles()

    panel.to_csv(os.path.join(OUT, "panel_full_with_scores.csv"), encoding="utf-8-sig")

    # 6) 关键数字汇总成 json，写报告和 PPT 时直接引用，省得手抄错
    import json

    def enc(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return None if not np.isfinite(o) else float(o)
        if isinstance(o, pd.Timestamp):
            return str(o.date())
        return str(o)

    summary = {
        "股票": "%s(%s)" % (STOCK_NAME, STOCK_CODE),
        "样本区间": "%s ~ %s" % (panel.index.min().date(), panel.index.max().date()),
        "交易日数": int(len(panel)),
        "回测样本天数": int(len(sample)),
        "最新信息": card.T.to_dict("records")[0],
        "回测绩效": perftab.to_dict("records"),
        "策略筛选": screen.to_dict("records"),
        "复合五分位": wide.reset_index().to_dict("records"),
        "NeweyWest": nw.to_dict("records"),
        "分时段": sub.to_dict("records"),
        "技术面统计": s["tech_stats"].to_dict(),
        "基本面统计": fund.stats.round(4).to_dict(),
        "CSMAR比率交叉验证": fund.cross_check.to_dict("records"),
    }
    with open(os.path.join(OUT, "run_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=1, default=enc)

    # 7) 屏幕上只打几条最关键的，细节都在 output/ 的 csv 里
    latest = card.T.to_dict("records")[0]
    bhrow = perftab[perftab["策略"] == "买入持有"].iloc[0]
    print("\n" + "-" * 70)
    print("最新一期 %s：技术面 %.1f / 基本面 %.1f / 复合 %.1f  ->  %s"
          % (latest["日期"], latest["技术面得分"], latest["基本面择时得分"],
             latest["复合得分"], latest["评级"]))
    print("  未来 1 个月：%s 个历史同分位样本，均值 %s%%，胜率 %s%%"
          % (latest["未来1个月_样本数"], latest["未来1个月_均值%"], latest["未来1个月_胜率%"]))
    print("  未来 3 个月：%s 个历史同分位样本，均值 %s%%，胜率 %s%%"
          % (latest["未来3个月_样本数"], latest["未来3个月_均值%"], latest["未来3个月_胜率%"]))
    print("  回测区间 %s ~ %s：买入持有年化 %.2f%%、最大回撤 %.2f%%"
          % (sample.index.min().date(), sample.index.max().date(),
             bhrow["年化收益%"], bhrow["最大回撤%"]))
    print("-" * 70)
    print("全部结果已写入 %s" % OUT)

    s.update({"bt": bt, "perftab": perftab, "card": card, "wide": wide,
              "nw": nw, "sub": sub, "screen": screen, "sample": sample})
    return s

In [16]:
# 一键跑完四个 stage（想分步看就把上面四行拆成四个 cell 单独执行）
s = stage_data()
s = stage_task1(s)
s = stage_task2(s)
s = stage_task3(s)

Step 0  读数据 -> 建日度面板
  [清洗] 东财不复权 3057 行，剔除成交量为 0 的行后剩 3057 行
  [清洗] 重复日期 0 条，已去重
  [清洗] 处理除权除息事件 14 个（含 5 次转增股本）
  [清洗] 触及涨停 53 天、触及跌停 36 天；收盘封死涨停 32 天、封死跌停 19 天；一字板 1 天
  [清洗] 总股本对齐到 3057 个交易日，缺失 0 天；最新总股本 57.7300 亿股
  [清洗] 非除权日 我的收益率 vs 交易所涨跌幅 最大绝对差 5.000e-05（应≈0）
  [清洗] 样本区间 2014-01-28 ~ 2026-09-30，共 3057 个交易日
  [清洗] 东财前复权早于 2019 出现非正价格 356 天，不可用于收益率计算（负价格陷阱）

Task 1  技术面择时指标
  [图] 已保存 C:\Users\35632\Desktop\python期中大作业\提交版\完整版代码及数据\output\task1_technical.png

Task 2  基本面择时指标与 PIT 对齐
  [基本面] 资产负债表：原始 118 行 -> 合并报表 64 行 -> 标准报告期 51 个（2013-12-31 ~ 2026-06-30），字段 13 个
  [基本面]   剔除非标准报告期 13 个：['2013-01-01', '2014-01-01', '2015-01-01', '2016-01-01', '2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01']
  [基本面] 利润表：原始 118 行 -> 合并报表 64 行 -> 标准报告期 51 个（2013-12-31 ~ 2026-06-30），字段 13 个
  [基本面]   剔除非标准报告期 13 个：['2013-01-01', '2014-01-01', '2015-01-01', '2016-01-01', '2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01',